# Two-loop $\phi^4$ renormalization with Feynkit and Rustred

This notebook reproduces the scalar self-energy calculation in the FeynCalc gallery's [two-loop $\phi^4$ renormalization example](https://feyncalc.github.io/FeynCalcExamples/Phi4/TwoLoops/Renormalization-SS), replacing its Kira reductions with `symbolica.community.hep`. The two-loop integrals are solved by Laporta elimination; the one-loop counterterm uses a parametric IBP recurrence.

We start from the example's scalar diagram amplitudes and carry out the Taylor expansion, integral reductions, UV-pole extraction, counterterm cancellation, and final $Z_\phi,Z_m$ checks. The analytic master integrals are inputs, as they are in the original example. FeynArts diagram generation is not required. Use a Python environment with the Rustred-enabled Symbolica Community `hep` extension and Jupyter installed.

In [1]:
from IPython.display import display
from symbolica import E, S
from symbolica.community import hep

d, k1, k2, M, p2, eps, g, pi = S("d", "k1", "k2", "M", "p2", "eps", "g", "pi")
T2, V, Lm, L4pi = S("T2", "V", "Lm", "L4pi")
zero = E("0")

def equal(left, right):
    """Exact equality of rational coefficients, including symbolic scales."""
    return (left - right).together().cancel().expand() == zero

## Diagram amplitudes and Taylor expansion

Write $M=m^2>0$ and $p^2=p_2$. With the common factor $i g^2$ suppressed, the diagrams are
\[
\frac1{4(k_1^2-M)^2(k_2^2-M)},\qquad
\frac1{6(k_1^2-M)(k_2^2-M)((k_1+k_2+p)^2-M)}.
\]
Their UV degree is two. Expanding the last denominator through $p^2$ and using Lorentz invariance, $\langle(q\cdot p)^2\rangle=p^2q^2/d$, gives
\[
\frac14 I_{210}+\frac16\left[I_{111}+p_2\left(\frac4d-1\right)I_{112}
+\frac{4Mp_2}{d}I_{113}\right],
\quad
I_{abc}=\int_{k_1,k_2}\frac1{D_1^aD_2^bD_3^c},
\]
where $D_1=k_1^2-M$, $D_2=k_2^2-M$, and $D_3=(k_1+k_2)^2-M$. The Taylor expansion preserves the UV poles because the massive denominators introduce no infrared singularity at $p=0$.

In [2]:
kinematics = hep.Kinematics(d, momenta=[k1, k2])
feynkit_family = hep.IntegralFamily(
    [k1, k2], [],
    [kinematics.scalar_product(k, k) - M for k in (k1, k2, k1 + k2)],
    kinematics=kinematics,
)
assert feynkit_family.is_complete and feynkit_family.is_independent
vacuum = hep.IBPFamily(feynkit_family, name="phi4_vacuum")
display(feynkit_family)

# Verify the entire denominator-permutation symmetry group by unit-Jacobian
# loop-momentum maps, before identifying equivalent residual master integrals.
symmetries = [
    feynkit_family.mapping_to(feynkit_family, images)
    for images in ([k1, k2], [k2, k1], [-k1-k2, k2],
                   [k1, -k1-k2], [k2, -k1-k2], [-k1-k2, k1])
]
assert all(mapping is not None for mapping in symmetries)
assert len({tuple(mapping.denominator_map) for mapping in symmetries}) == 6

def canonical(powers):
    return min(tuple(mapping.map_powers(powers)) for mapping in symmetries)


D1,"-M+dot(k1(mink(d)),k1(mink(d)))"
D2,"-M+dot(k2(mink(d)),k2(mink(d)))"
D3,"-M+dot(k1(mink(d)),k1(mink(d)))+2·dot(k1(mink(d)),k2(mink(d))) +dot(k2(mink(d)),k2(mink(d)))"


## Laporta reduction of the two-loop integrals

The four targets are exactly the scalar integrals needed above. We use $T^2=I_{110}$ and $V=I_{111}$ as master integrals. Independent identities provide checks on the computed rules: one-loop scaling fixes $I_{210}$, common-mass differentiation and permutation symmetry fix $I_{112}$, and the two contractions $\partial_{k_1}\cdot k_{1,2}$ at $(1,1,2)$ fix $I_{113}$.

In [3]:
targets = [[2, 1, 0], [1, 1, 1], [1, 1, 2], [1, 1, 3]]
laporta = vacuum.reduce_laporta(targets=targets, max_depth=2)

# The three disconnected routings all factor into the same T*T.
masters = {canonical([1, 1, 0]): T2, canonical([1, 1, 1]): V}
reduced = {}
for target in targets:
    terms = laporta.reduce(target)
    assert all(canonical(powers) in masters for powers, _ in terms), terms
    reduced[tuple(target)] = sum(
        (coefficient * masters[canonical(powers)] for powers, coefficient in terms), zero
    ).together().cancel()
    display((target, reduced[tuple(target)]))

assert equal(reduced[2, 1, 0], (d - 2) * T2 / (2 * M))
assert equal(reduced[1, 1, 2], (d - 3) * V / (3 * M))
assert equal(reduced[1, 1, 3],
             (d - 8) * (d - 3) * V / (18 * M**2)
             + (d - 2)**2 * T2 / (12 * M**3))

([2, 1, 0], 1/2·(d·T2-2·T2)/M)

([1, 1, 1], V)

([1, 1, 2], 1/3·(d·V-3·V)/M)

([1, 1, 3], 1/36·(-22·d·M·V-12·d·T2+48·M·V+12·T2+2·d²·M·V+3·d²·T2)/M³)

In [4]:
bare_integrals = (
    reduced[2, 1, 0] / 4
    + (reduced[1, 1, 1]
       + p2 * (4 / d - 1) * reduced[1, 1, 2]
       + 4 * M * p2 / d * reduced[1, 1, 3]) / 6
).together().cancel()
display(bare_integrals)

-1/216·(4·d·M·p2·V+54·d·M·T2+48·d·p2·T2-36·d·M²·V-48·M·p2·V-48·p2·T2+4·d²·M·p2·V
    -27·d²·M·T2-12·d²·p2·T2
)/(d·M²)

## Analytic masters and UV poles

In the original example's normalization, the master inputs are
\[
T=-e^{\gamma_E\epsilon}M^{1-\epsilon}\Gamma(\epsilon-1)
=M\left[\frac1\epsilon+1-\log M+O(\epsilon)\right],
\]
\[
V=M\left[\frac{3}{2\epsilon^2}+\frac{9/2-3\log M}{\epsilon}+O(1)\right].
\]
These follow from FeynCalc's [tadpole master](https://raw.githubusercontent.com/FeynCalc/feyncalc/master/FeynCalc/Examples/MasterIntegrals/Tadpoles/tad1LxFx1x1xxEp999x.m) and [equal-mass vacuum master](https://raw.githubusercontent.com/FeynCalc/feyncalc/master/FeynCalc/Examples/MasterIntegrals/Tadpoles/tad2LxFx111x111xxEp1x.m). Only their poles and the tadpole's finite term are needed; the reduced coefficients are regular at $\epsilon=0$.

Set $d=4-2\epsilon$, $L_m=\log M$, and $L_{4\pi}=\log(4\pi)$. Each loop contributes $i(4\pi)^{\epsilon-2}$. Below amplitudes are divided by $i g^2/(16\pi^2)^2$, so the two-loop normalization contributes $-(1+2\epsilon L_{4\pi})$.

In [5]:
T_poles = M * (1 / eps + 1 - Lm)
T2_poles = M**2 * (1 / eps**2 + 2 * (1 - Lm) / eps)
V_poles = M * (E("3/2") / eps**2 + (E("9/2") - 3 * Lm) / eps)

bare_series = (
    -(1 + 2 * eps * L4pi)
    * bare_integrals.replace(d, 4 - 2 * eps).replace(T2, T2_poles).replace(V, V_poles)
).series(eps, 0, -1)
bare_uv = bare_series.to_expression().expand()
expected_bare = -M / (2 * eps**2) + (M * (Lm - 1 - L4pi) + p2 / 24) / eps
assert equal(bare_uv, expected_bare)
display(bare_uv)

-M/eps+M·Lm/eps-M·L4pi/eps-1/2·M/eps²+1/24·p2/eps

## Parametric IBP reduction of the counterterm

For $T(a)=\int_k(k^2-M)^{-a}$, the identity $\partial_k\cdot k$ gives
\[
T(a+1)=\frac{d-2a}{2aM}T(a),\qquad a>0.
\]
We ask Rustred to derive this recurrence with symbolic $a$ and specialize it to $T(2)$. The one-loop renormalization coefficients in units of $g/(16\pi^2)$ are $z_g^{(1)}=3/(2\epsilon)$ and $z_m^{(1)}=1/(2\epsilon)$, while $z_\phi^{(1)}=0$.

The order-$g^2$ counterterm integrand is $\tfrac12[z_g^{(1)}T(1)+Mz_m^{(1)}T(2)]$. Its single loop has the normalization factor $1+\epsilon L_{4\pi}$ in the same amplitude units as above.

In [6]:
tadpole_kinematics = hep.Kinematics(d, momenta=[k1])
tadpole_family = hep.IntegralFamily(
    [k1], [], [tadpole_kinematics.scalar_product(k1, k1) - M],
    kinematics=tadpole_kinematics,
)
tadpole = hep.IBPFamily(tadpole_family, name="phi4_tadpole")
parametric = tadpole.solve_parametric(sector=[True], max_depth=1)
display([(rule.target, rule.terms, rule.nonzero_conditions)
         for rule in parametric.rules])
tadpole_terms = parametric.reduce([2])
assert len(tadpole_terms) == 1 and tuple(tadpole_terms[0][0]) == (1,)
tadpole_coefficient = tadpole_terms[0][1]
assert equal(tadpole_coefficient, (d - 2) / (2 * M))

[([n1], [([-1+n1], (2+d-2·n1)/(-2·M+2·M·n1))], [-2·M+2·M·n1, 1])]

In [7]:
zg1 = 3 / (2 * eps)
zm1 = 1 / (2 * eps)
counterterm_integrals = (
    zg1 + M * zm1 * tadpole_coefficient.replace(d, 4 - 2 * eps)
) * T_poles / 2
counterterm_uv = (
    (1 + eps * L4pi) * counterterm_integrals
).series(eps, 0, -1).to_expression().expand()
expected_counterterm = M / eps**2 + M * (E("3/4") - Lm + L4pi) / eps
assert equal(counterterm_uv, expected_counterterm)
loop_uv = (bare_uv + counterterm_uv).expand()
assert equal(loop_uv, M / (2 * eps**2) - M / (4 * eps) + p2 / (24 * eps))
display(counterterm_uv, loop_uv)

3/4·M/eps-M·Lm/eps+M·L4pi/eps+M/eps²

-1/4·M/eps+1/2·M/eps²+1/24·p2/eps

## Renormalization constants

$Z_m$ multiplies the mass-squared parameter, matching `Zmphi` in the source example. The tree counterterm at two loops is $p_2z_\phi^{(2)}-M(z_m^{(2)}+z_\phi^{(2)})$ in our amplitude units. Its coefficients must cancel the loop poles. With $a=g/(16\pi^2)$ this determines
\[
Z_\phi=1-\frac{a^2}{24\epsilon},\qquad
Z_m=1+\frac{a}{2\epsilon}+a^2\left(\frac1{2\epsilon^2}-\frac5{24\epsilon}\right).
\]
The following assertions compare the reconstructed expressions with the original gallery result and verify cancellation of both $p^2$ and mass poles, including all logarithms.

In [8]:
zphi2 = -loop_uv.coefficient(p2).expand()
zm2 = (loop_uv.replace(p2, 0) / M - zphi2).expand()
assert equal(loop_uv + p2 * zphi2 - M * (zm2 + zphi2), zero)

a = g / (16 * pi**2)
Zphi = (1 + a**2 * zphi2).expand()
Zm = (1 + a * zm1 + a**2 * zm2).expand()
assert equal(Zphi, 1 - g**2 / (6144 * pi**4 * eps))
assert equal(Zm, 1 + g / (32 * pi**2 * eps)
             + g**2 / (512 * pi**4 * eps**2) - 5 * g**2 / (6144 * pi**4 * eps))
display(Zphi, Zm)
print("Both Kira replacements and the full two-loop renormalization result agree.")

1-1/6144·g²/(𝜋⁴·eps)

1+1/32·g/(𝜋²·eps)-5/6144·g²/(𝜋⁴·eps)+1/512·g²/(𝜋⁴·eps²)

Both Kira replacements and the full two-loop renormalization result agree.
